# Phase 8: healing the 30-block student

**Goal:** remove four of the 32 blocks of Qwen-Image 2.1, keeping spelling, which is the first capability to break.

**Starting point (from Phase 7, `phase7-second-block-removal.ipynb`):** the 30-block student is B (block 5 removed) with
both removed blocks replaced by attention bridges of the same architecture (timestep-conditioned rank-1024 MLP plus an
8-head attention branch with its own prefix cache, about 25.2M weights each):

| Slot | Replacement | Chosen checkpoint |
|---|---|---|
| 5 | attention bridge, trained end to end with the block-20 bridge frozen | step 200 |
| 20 | attention bridge, trained end to end inside B | step 600 |

It spells 10 of 11 legible signs on the four-seed text grid, and its velocity error is 8.09%.

**Plan:** (1) heal the 30-block student with a LoRA on its 30 blocks, trained jointly with both bridges (option B);
(2) merge the LoRA; (3) Phase 9 chooses removals three and four with a new text-aware scan inside the healed student;
(4) the speed-distillation (lightning) LoRA is trained last. Every run is judged by velocity and by images and spelling,
and every checkpoint is saved. Architecture: `niwaki-page/nihonga/student30-architecture.svg`.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
assert (ROOT / 'PLAN.md').exists(), 'Run this notebook from the repository root.'
sys.path.insert(0, str(ROOT / 'tools'))
for stream in (sys.stdout, sys.stderr):      # Modal prints non-ASCII progress symbols
    if hasattr(stream, 'reconfigure'):
        stream.reconfigure(encoding='utf-8')
ALLOW_GPU = False          # set True only to launch a step whose saved result is missing
SOURCES = {}               # worker sources, embedded per step exactly as executed


def load_summary(run):
    path = ROOT / run / 'summary.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def reuse_or_launch(run, launch):
    summary = load_summary(run)
    if summary is not None:
        print('Saved result:', run, '| status', summary.get('status'))
        return summary
    if (ROOT / run).parent.joinpath('aborted.json').exists():
        print('Run stopped by request; nothing to reuse.')
        return None
    if not ALLOW_GPU:
        raise RuntimeError(f'No saved result for {run}; set ALLOW_GPU = True to launch it on Modal.')
    launch()
    return load_summary(run)


def pct(value, digits=2):
    return f'{100 * value:.{digits}f}%'


def velocity_row(label, block):
    o = block['overall']['teacher_relative_l2']
    p = {k: block['panels'][k]['teacher_relative_l2']['mean'] for k in block['panels']}
    s = {k: block['stages'][k]['teacher_relative_l2']['mean'] for k in block['stages']}
    return (f"| {label} | {pct(o['mean'])} | {pct(p['gate10'])} | {pct(p['wider50'])} | "
            f"{pct(s['early'], 1)} / {pct(s['middle'], 1)} / {pct(s['late'], 1)} | {pct(o['p90'], 1)} | {pct(o['maximum'], 1)} |")


def velocity_table(rows):
    head = '| Model | Overall | Check (10) | Wider (50) | Early / middle / late | p90 | Worst |\n|---|---|---|---|---|---|---|\n'
    display(Markdown(head + '\n'.join(rows)))


def show_variants(summary):
    rows = [velocity_row(f"{name} (step {v['fit']['selected_step']}, {v['parameters'] / 1e6:.1f}M params)", v)
            for name, v in summary['variants'].items()]
    velocity_table(rows)


def show_images(folder, names):
    for name in names:
        display(Image(filename=str(ROOT / folder / name), width=1000))

## 1. Healing, option B

A rank-16 LoRA on q, k, v and out of each of the 30 original blocks (15.7M weights, lr 2e-5) is trained together with
both bridges (50.4M weights, lr 1e-5) against the teacher's final velocity, on the saved recordings (450 text prompts)
and the original training data. Original weights stay frozen. Every 50 updates the student is saved and measured on the
180 velocity comparisons; the text grids follow for every checkpoint (`data/phase8/healing/protocol.json`).

In [ ]:
# Healing: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['healing_worker.py'] = r'''def heal_student(request, key):
    """LoRA healing pilot of C = B + block-13 attention bridge, against the final velocity.

    Verifies the teacher contract and exact B replay, computes live teacher targets, loads the saved text-rich
    recordings, then trains LoRA adapters on every original block together with both bridges.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase8/healing') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # ---- Live teacher targets for TRAIN inputs (stored legacy targets differ by up to ~2.6%). ----
        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_full, early_pool = [], []
        for record in request['records']:
            if not record['capture_id'].startswith('p5-train-'):
                continue
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            if item['row']['split'] != 'train' or item['row']['source_id'] != record['source_id']:
                raise RuntimeError('Training record identity differs.')
            train_full.append(item)
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_full.append(item)
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            early_pool.append(item)
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        train_ids = {i['row']['source_id'] for i in train_full} | {i['row']['source_id'] for i in early_pool}
        if len(train_full) != 60 or train_ids & validation_ids:
            raise RuntimeError('Training set size or split differs.')
        for item in train_full + early_pool:
            cache = QwenImage21KVCache(32)
            item['live_targets'] = []
            for stage in item['stages']:
                with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    item['live_targets'].append(forward(item, stage, cache).to(torch.bfloat16))
                report['teacher_calls'] += 1
        report['train_counts'] = {'full_prompts': len(train_full), 'early_pool': len(early_pool)}
        print('Live teacher targets for', report['train_counts'], flush=True)

        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- LoRA healing pilot: C = B + block-13 attention bridge; LoRA on every original block, both bridges trainable.
        import random
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        fit = request['fit']
        lora = request['lora']
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)

        # Saved recordings from the 8-head bridge run (steps 0 and 19 with teacher targets).
        recordings = request['recordings']
        new_text, new_general = [], []
        for row in recordings['rows']:
            item = torch.load(Path(recordings['remote_directory']) / (row['source_id'] + '.pt'), weights_only=True, map_location='cpu')
            if (item['row']['source_id'] != row['source_id'] or [s['stage'] for s in item['stages']] != ['early', 'middle']
                    or len(item['live_targets']) != 2):
                raise RuntimeError('Saved recording differs: ' + row['source_id'])
            (new_text if row['subdimension'] == 'text_rendering' else new_general).append(item)
        if {i['row']['source_id'] for i in new_text + new_general} & (train_ids | validation_ids):
            raise RuntimeError('Recordings overlap earlier training or validation prompts.')
        report['recording_counts'] = {'text': len(new_text), 'general': len(new_general)}
        general_pool = train_full + new_general

        # Both attention bridges of the 30-block student, FP32 and trainable: block 5 (step 200) and block 20 (step 600).
        spec = {}
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=True)
            spec[bridge_spec['layer']] = dict(variant, step=bridge_spec['step'])
            del checkpoint
        # LoRA on q, k, v and out of every original block, then functional checkpointing for memory.
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in (5, 20):
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        lora_parameters = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        bridge_parameters = {n: p for n, p in model.named_parameters() if n.startswith(('transformer_blocks.5.bridge.', 'transformer_blocks.20.bridge.'))}
        if len(lora_parameters) != 30 * 4 * 2 or not bridge_parameters:
            raise RuntimeError('Adapter structure differs.')
        trainable = {**lora_parameters, **bridge_parameters}
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = {'lora': sum(p.numel() for p in lora_parameters.values()),
                                          'bridges': sum(p.numel() for p in bridge_parameters.values())}
        model.eval()
        def train_loss(item):
            cache = QwenImage21KVCache(32)
            losses = []
            for stage, target in zip(item['stages'], item['live_targets']):
                full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
                kwargs = gpu({field: full[field] for field in argument_order})
                kwargs['kv_cache'] = cache
                prediction = model(**kwargs)[0][:, -stage['target_image_tokens']:].float()
                target = target.to('cuda').float()
                losses.append((prediction - target).square().mean() / target.square().mean().clamp_min(1e-12))
                report['student_calls'] += 1
            loss = torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        optimizer = torch.optim.AdamW([{'params': list(lora_parameters.values()), 'lr': lora['learning_rate'], 'base': lora['learning_rate']},
                                       {'params': list(bridge_parameters.values()), 'lr': fit['bridge_learning_rate'], 'base': fit['bridge_learning_rate']}],
                                      weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        report['checkpoints'] = []
        started = time.perf_counter()

        def evaluate_and_save(step):
            # Quantitative test on all 180 comparisons, and a saved checkpoint for the qualitative (image) test.
            for parameter in trainable.values():
                parameter.requires_grad_(False)
            rows = predictions()
            for parameter in trainable.values():
                parameter.requires_grad_(True)
            report['student_calls'] += len(rows)
            records = metrics(rows)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         overall=aggregate(records),
                         panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                         stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')})
            if step > 0:
                path = directory / f'healing_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'lora': lora, 'bridge_variants': spec,
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            o = entry['overall']['teacher_relative_l2']
            print(f"healing step {step}: overall {o['mean']:.6f} gate10 {entry['panels']['gate10']['teacher_relative_l2']['mean']:.6f} "
                  f"wider {entry['panels']['wider50']['teacher_relative_l2']['mean']:.6f} ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                evaluate_and_save(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = group['base'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    pool = new_text if rng.random() < fit['text_fraction'] else general_pool
                    (train_loss(rng.choice(pool)) * (1 - fit['early_weight']) / fit['accumulation']).backward()
                    (train_loss(rng.choice(early_pool)) * fit['early_weight'] / fit['accumulation']).backward()
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
        report['status'] = 'passed'
        report['scope'] = 'LoRA healing of C with every checkpoint saved and evaluated on all 180 velocity comparisons; images are rendered per checkpoint in a separate step.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    protocol_path = root / 'data/phase8/healing/protocol.json'
    v2 = load(next((root / 'data/phase7/healing_pilot_v2').glob('*/request.json')).relative_to(root))['request']

    def bridge(run_folder, slot, step):
        run = load(next((root / run_folder).glob('*/summary.json')).relative_to(root))
        checkpoint = next(c['checkpoint'] for c in run['checkpoints'] if c['step'] == step)
        return dict(layer=slot, step=step, remote_path=f"/phase3/phase7/block_bridge/{run['request_key']}/{checkpoint['name']}",
                    sha256=checkpoint['sha256'])

    request = {k: v for k, v in v2.items() if k not in ('worker_source', 'warm_start', 'gate_references', 'gate_rule', 'scope', 'version', 'selection')}
    request.update(
        version='phase8-healing-v1', worker_source=SOURCES['healing_worker.py'],
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        bridges=[bridge('data/phase7/block5_bridge', 5, 200), bridge('data/phase7/block20_bridge', 20, 600)],
        lora=dict(rank=16, alpha=16, learning_rate=2e-5, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261018, sampling_seed=20261101, bridge_learning_rate=1e-5, warmup=10, gradient_clip=1.0,
                 updates=200, checkpoint_every=50, accumulation=2, early_weight=0.3, text_fraction=0.4),
        selection='No automatic selection: every checkpoint is saved, gets the full velocity test and the four-seed text grid; spelling decides',
        scope='Healing option B of the 30-block student (blocks 5 and 20 bridged): LoRA on the 30 blocks plus both bridges')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase8/healing' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase8/healing/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase8-healing')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'healing_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['heal_student'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'recording_counts', 'trainable_parameters')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase8/healing/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    velocity_table([velocity_row(f"step {c['step']}", c) for c in summary['checkpoints']])

### What the velocity results mean

| Checkpoint | Overall | Check (10) | Wider (50) | Late moment |
|---|---|---|---|---|
| step 0 (30 blocks, unhealed) | 8.09% | 8.50% | 8.01% | 8.1% |
| step 50 | 8.01% | 8.36% | 7.94% | 8.1% |
| step 100 | 7.97% | 8.33% | 7.90% | 8.0% |
| step 150 | 7.90% | 8.28% | 7.82% | 8.0% |
| step 200 | **7.84%** | 8.26% | 7.76% | 8.0% |

Step 0 reproduces the unhealed student exactly, so both bridges were loaded as chosen. Velocity improves steadily
and is still falling at step 200, most at the early moment (11.8% to 11.2%). The gain is small, as expected from the
gentle learning rates. Velocity cannot see letters, so spelling decides (step 1b).

## 1b. Spelling of every healing checkpoint

The 5 text prompts with 4 seeds for every saved checkpoint (steps 50 to 200), beside the teacher, B and the unhealed
30-block student.

In [ ]:
# Healing text grids: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
IMAGE_ROWS, IMAGE_STEPS = 'text', [c['step'] for c in summary['checkpoints'] if c['step'] > 0] if summary else []
SOURCES['healing_images_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase8/healing_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [c['name'] for c in request['checkpoints']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = False
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridges = {}
        lora_modules = {}
        for checkpoint_spec in request['checkpoints']:
            healed = load_checkpoint(checkpoint_spec)
            state = healed['state_dict']
            lora = healed['lora']
            if not bridges:
                # Both attention bridges (replacing the B block-5 bridge inserted above), then LoRA on the 30 original blocks.
                for layer, variant in sorted(healed['bridge_variants'].items()):
                    bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
                    bridge.requires_grad_(False)
                    conditioned['insert_conditioned_bridge'](model, int(layer), bridge.float(), keep_dtype=True)
                    bridges[int(layer)] = model.transformer_blocks[int(layer)].bridge
                if sorted(bridges) != [5, 20]:
                    raise RuntimeError('Bridge slots differ.')
                for slot, block in enumerate(model.transformer_blocks):
                    if slot in bridges:
                        continue
                    for name in ('to_q', 'to_k', 'to_v'):
                        setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
                    block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
                    for name, module in (('to_q', block.attn.to_q), ('to_k', block.attn.to_k), ('to_v', block.attn.to_v), ('to_out.0', block.attn.to_out[0])):
                        lora_modules[(slot, name)] = module
                model.requires_grad_(False)
            for layer, bridge in bridges.items():
                prefix = f'transformer_blocks.{layer}.bridge.'
                bridge.load_state_dict({k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)})
            loaded = 0
            for (slot, name), module in lora_modules.items():
                for part in ('down', 'up'):
                    getattr(module, part).weight.data.copy_(state[f'transformer_blocks.{slot}.inner.attn.{name}.{part}.weight'])
                    loaded += 1
            if loaded != 30 * 4 * 2:
                raise RuntimeError('Healed adapter count differs.')
            model.eval()
            for row in request['rows']:
                outputs[checkpoint_spec['name']][row['source_id']] = generate(row, checkpoint_spec['name'])
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_text():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    heal = load(RUN + '/summary.json')
    heal_request = load(RUN + '/request.json')['request']
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                              for r in text_rows for k in (1, 2, 3)]
    if IMAGE_ROWS == 'general':
        rows = [r for r in base['rows'] if r['subdimension'] not in ('text_rendering', 'posters')]
    checkpoints = [dict(name=f"healed_s{c['step']:03d}", step=c['step'],
                        remote_path=f"/phase3/phase8/healing/{heal['request_key']}/{c['checkpoint']['name']}",
                        sha256=c['checkpoint']['sha256']) for c in heal['checkpoints'] if c['step'] in IMAGE_STEPS]
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase8-healing-images-v1', rows=rows, worker_source=SOURCES['healing_images_worker.py'],
        conditioned_bridge_source=heal_request['conditioned_bridge_source'], components_source=heal_request['components_source'],
        execution_account='new', candidate_only=True, checkpoints=checkpoints,
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name='unused'),
        scope='Healed 30-block student: ' + ('4-seed text grid for every saved checkpoint' if IMAGE_ROWS == 'text' else 'general prompts'))
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase8/healing_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase8/healing_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase8-healing-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'healing_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

def fetch_images(run):
    """Download a rendered run's images from the Modal volume into .archive, checking every hash."""
    import hashlib
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    store = ROOT / '.archive/phase8_healing_images' / key
    store.mkdir(parents=True, exist_ok=True)
    for artifact in json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))['artifacts']:
        path = store / artifact['name']
        if not path.exists():
            path.write_bytes(b''.join(volume.read_file(f"/phase8/healing_images/{key}/{artifact['name']}")))
        if hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            raise RuntimeError('Image hash mismatch: ' + artifact['name'])
    return store


def build_healing_text_grids(run_text):
    """Rows: teacher, B, unhealed 30-block student (new block-5 bridge step 200), then every healed checkpoint; seeds 0-3."""
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    store = fetch_images(run_text)
    names = [c['name'] for c in json.loads((ROOT / run_text / 'request.json').read_bytes())['request']['checkpoints']]
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    unhealed = ROOT / '.archive/block5_bridge_images/3c2612021d83a666cc8e802b0f156e5e10d353f02d337f7c8e2ee46288766634'
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 21)
    out = ROOT / run_text / 'text_sheets'
    out.mkdir(exist_ok=True)
    tile, label = 300, 330
    for row in [r for r in first['rows'] if r['subdimension'] in ('text_rendering', 'posters')]:
        sid = row['source_id']
        name = lambda k, suffix: f'{sid}-{suffix}.png' if k == 0 else f'{sid}-s{k}-{suffix}.png'
        models = [('Teacher (32)', lambda k: first_store / row['teacher']['name'] if k == 0 else reference / name(k, 'teacher')),
                  ('B (31)', lambda k: first_store / row['B']['name'] if k == 0 else reference / name(k, 'B')),
                  ('30 blocks, unhealed', lambda k: unhealed / name(k, 'bridge5_s200'))]
        models += [(f"30 blocks, healed step {int(n.split('_s')[1])}", lambda k, n=n: store / name(k, n)) for n in names]
        canvas = PILImage.new('RGB', (label + 4 * tile, 80 + len(models) * tile), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
        for column in range(4):
            draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
        for line, (title, locate) in enumerate(models):
            y = 80 + line * tile
            draw.text((10, y + tile // 2 - 12), title, font=big, fill='black')
            for column in range(4):
                with PILImage.open(locate(column)) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        canvas.save(out / f'{sid}.jpg', quality=88)
    return out

LAUNCH_ARGS = ['--dry-run']
RUN_TEXT = 'data/phase8/healing_images/' + main_text()
LAUNCH_ARGS = []
summary_text = reuse_or_launch(RUN_TEXT, main_text)
if summary_text and summary_text.get('status') == 'passed':
    sheets = build_healing_text_grids(RUN_TEXT)
    show_images(str(sheets.relative_to(ROOT)), sorted(p.name for p in sheets.glob('*.jpg')))